# Induction heads in the wild

In [*I Didn't Understand QKV, So I Hand-Crafted an Induction Head*](https://barmag.github.io/mechanistic-interpretability/machine-learning/learning-in-public/2026/07/15/i-didnt-understand-qkv-so-i-hand-crafted-an-induction-head.html) we built a two-head induction circuit by hand: a **previous-token head** whose QK was a literal `shift` matrix, feeding a **K-composed induction head** whose QK matched token identity and whose OV copied. Eight matrices, no training, 87% correct.

The post ended on a question, and this notebook answers it:

> If I take a small open-weights model like Pythia and go looking for its previous-token head and its induction head, will the weights look anything like the shift and projection I just wrote by hand? Or does training find some smeared version, spread across heads, that only approximates this behavior?

**Hypothesis.** The hand-crafted circuit is reproducible in pre-trained models:

1. GPT-2 small's previous-token head implements `shift` visibly in its **positional QK circuit** (a subdiagonal stripe).
2. Both models' induction heads show **token-identity QK matching** (through composition with the prev-token head), a **copying OV**, and a **K-composition score** that singles out the prev-token head.
3. **Falsifiable twist:** in Pythia-160m the `shift` matrix should *not exist as a weight-space object* — rotary embeddings (RoPE) never write position into the residual stream, so the same behavior must be implemented without the matrix the toy used.

**Models:** GPT-2 small (learned absolute positions — the `shift` question is answerable matrix-by-matrix) and Pythia-160m (RoPE — the contrast case).

**Papers:** Olsson et al. 2022, [*In-context Learning and Induction Heads*](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html) (behavioral scores); Elhage et al. 2021, [*A Mathematical Framework for Transformer Circuits*](https://transformer-circuits.pub/2021/framework/index.html) (QK/OV circuits, composition scores).

In [1]:
import os
os.environ.setdefault("HSA_OVERRIDE_GFX_VERSION", "11.0.0")  # Strix Halo gfx1151 runs the gfx1100 wheels

import subprocess

import matplotlib.pyplot as plt
import numpy as np
import torch

SEED = 42
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"device={device}  seed={SEED}  commit={commit}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

device=cuda  seed=42  commit=9d180ec
GPU: AMD Radeon 8060S Graphics


## Act 0 — The toy, re-run

Before opening any trained weights, here is the object of comparison, verbatim from the post: a 15-dimensional residual stream split into three blocks (`what I am`, `before me`, `where I am`), layer 1's QK a `shift` permutation that asks "who sits at position i−1?", layer 2's QK an identity match against the `before me` block, and an OV that copies. Every matrix is exact because we chose it; the question for the rest of the notebook is which of these choices training rediscovers.

The four jobs we will hunt, and where they live in the toy:

| Job | Toy matrices | What it does |
|---|---|---|
| shift-QK | `W_Q1 = shift×3`, `W_K1 = I×3` (position block) | attend to position i−1 |
| token-match-QK | `W_Q2 = I×2` (token block), `W_K2 = I×2` (before-me block) | "who has *my* token in their before-me slot?" |
| K-composition | `W_O1` writes what `W_K2` reads | layer 1's output becomes layer 2's key |
| copy-OV | `W_V2 = I×4`, `W_O2 = I` | hand over the matched position's own token |

In [2]:
import numpy as np

VOCAB = ["apple", "banana", "cherry", "kiwi", "lemon"]
TOK = {t: i for i, t in enumerate(VOCAB)}
D_MODEL = 15  # 5 "what I am" + 5 "before me" + 5 "where I am"

def embed(seq):
    x = np.zeros((len(seq), D_MODEL))
    for i, tok in enumerate(seq):
        if tok is not None:
            x[i, TOK[tok]] = 1.0   # what I am
        x[i, 10 + i] = 1.0         # where I am
    return x

def attention(x, w_q, w_k, w_v, w_o):
    q, k, v = x @ w_q.T, x @ w_k.T, x @ w_v.T
    scores = q @ k.T
    scores[np.triu_indices(len(x), k=1)] = -np.inf   # causal mask
    scores -= scores.max(axis=1, keepdims=True)
    attn = np.exp(scores)
    attn /= attn.sum(axis=1, keepdims=True)
    return x + (attn @ v) @ w_o.T, attn

# --- Layer 1: previous-token head ---
shift = np.zeros((5, 5))
for i in range(1, 5):
    shift[i - 1, i] = 1.0   # onehot(i) -> onehot(i-1)

W_Q1 = np.hstack([np.zeros((5, 10)), shift * 3])
W_K1 = np.hstack([np.zeros((5, 10)), np.eye(5) * 3])
W_V1 = np.hstack([np.eye(5), np.zeros((5, 10))])
W_O1 = np.vstack([np.zeros((5, 5)), np.eye(5), np.zeros((5, 5))])

# --- Layer 2: induction head ---
W_Q2 = np.hstack([np.eye(5) * 2, np.zeros((5, 10))])
W_K2 = np.hstack([np.zeros((5, 5)), np.eye(5) * 2, np.zeros((5, 5))])
W_V2 = np.hstack([np.eye(5) * 4, np.zeros((5, 10))])
W_O2 = np.vstack([np.eye(5), np.zeros((10, 5))])

def predict_next(seq):
    x = embed(seq)
    x, _ = attention(x, W_Q1, W_K1, W_V1, W_O1)
    x, attn2 = attention(x, W_Q2, W_K2, W_V2, W_O2)
    logits = x[-1, :5]
    probs = np.exp(logits) / np.exp(logits).sum()
    return dict(zip(VOCAB, probs.round(4))), attn2[-1].round(4)

for seq in [[None, "banana", "apple", "cherry", "banana"],
            [None, "apple", "banana", "cherry", "apple"]]:
    probs, attn = predict_next(seq)
    top = max(probs, key=probs.get)
    shown = " ".join(t if t else "<s>" for t in seq)
    print(f"{shown} -> {top} ({probs[top]:.1%})")

<s> banana apple cherry banana -> apple (87.0%)
<s> apple banana cherry apple -> banana (87.0%)
